In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys
sys.path.append('/media/dll-1/SSD 4TB/code/REEDNet/')  # Replace with real path
from EEGDataLoader import EEGDataset
from models.REEDNetModel import REEDNetModel
import config
from train_eval import eval, train

model_saving_name = "/media/dll-1/SSD 4TB/EEG Datasets/experiment/nouman_conv_2/model/weights/conv_tt_0_original.pt"

epochs = 50
batch_size = 64
hidden_size = 20
output_size = 3  # Binary classification
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [0]

In [ ]:
train_dataset = EEGDataset(config.train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(config.val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(config.test_data_csv, number_of_eeg_channels=number_of_eeg_channels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [ ]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = REEDNetModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.Size([1, 1, 400]) torch.Size([1, 33, 14]) torch.Size([1, 30, 400])
Output shape: torch.Size([1, 3])


ConvParallelEEG1DModel(
  (branches): ModuleList(
    (0): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(in_features=32, out_features=3, bias=True)
    )
    (1): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(33, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(

In [ ]:
prev_eval_loss = 0
prev_eval_acc = 0

13132


In [16]:
model.eval()
# total trainable params
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {params:,}  (~{params*4/1e6:.2f} MB in FP32)")  # 4 bytes/param

Params: 13,132  (~0.05 MB in FP32)


In [8]:
model = model.to(device)

prev_eval_loss, prev_eval_acc = eval(model, val_loader, loss_fn, 1,False,'', active_branches, device)
print("Loss:",prev_eval_loss, "Accuracy",prev_eval_acc)

Eval:   0%|          | 0/43343 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x7ee769e9c700>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval:   6%|▌         | 2685/43343 [00:11<02:49, 239.47it/s]


KeyboardInterrupt: 

In [ ]:
result = {"epoch":[],"train_loss":[],"train_acc":[],"eval_loss":[],"eval_acc":[],"test_acc":[],"test_loss":[]}
saved_epoch = 0

In [ ]:
for epoch in range(epochs):  # loop over the dataset multiple times
    train_loss, train_acc = train(model, train_loader,optimizer, loss_fn, active_branches, device)
    print(f"Train Loss after epoch {epoch+2}: {train_loss} Train ACC after epoch {epoch}: {train_acc}")
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    print(f"VAL Loss after epoch {epoch}: {eval_loss} VAL ACC after epoch {epoch}: {eval_acc}")
    # test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    test_loss = eval_loss
    test_acc = eval_acc
    print(f"Test Loss after epoch {epoch}: {test_loss} Test Acc after epoch {epoch}: {test_acc}")

    scheduler.step(eval_loss)
    
    result['epoch'].append(epoch)
    
    result['train_loss'].append(train_loss)
    result['train_acc'].append(train_acc)
    result['eval_loss'].append(eval_loss)
    result['eval_acc'].append(eval_acc)
    result['test_loss'].append(test_loss)
    result['test_acc'].append(test_acc)

    if prev_eval_acc < eval_acc:
      prev_eval_acc = eval_acc
      saved_epoch = epoch
      torch.save(model.state_dict(), model_saving_name)
      print("WEIGHTS SAVED")

Training:   0%|          | 0/4531 [00:00<?, ?it/s]

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Correct Classified: 183474.0 / 289972
Class-wise correct: Class 0 = 123603.0, Class 1 = 19495.0, Class 2 = 40376.0
Train Loss after epoch 2: 1.836254233688846 Train ACC after epoch 0: 0.6327300566951292


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


...

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Correct Classified: 217292.0 / 289972
Class-wise correct: Class 0 = 121890.0, Class 1 = 33032.0, Class 2 = 62370.0
Train Loss after epoch 51: 1.6145632598763715 Train ACC after epoch 49: 0.749355110148566


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


Correct classified: 60526.0 / 74774
Class-wise correct: Class 0 = 32064.0, Class 1 = 9072.0, Class 2 = 19390.0
Precision: 76.75  Recall: 78.00  F1: 77.27
Confusion Matrix:
 [[32064  1408  3091]
 [ 1474  9072  2626]
 [ 1173  4476 19390]]
VAL Loss after epoch 49: 0.6303254231301602 VAL ACC after epoch 49: 0.8094524834835638
Test Loss after epoch 49: 0.6303254231301602 Test Acc after epoch 49: 0.8094524834835638


In [ ]:
print("Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  ")
for epoch in range(epochs):
    if(saved_epoch == epoch):
        print("Saved Weights")
    print(result['epoch'][epoch],"|",result['train_loss'][epoch],"|",result['train_acc'][epoch],"|",result['eval_loss'][epoch],"|",result['eval_acc'][epoch],"|",result['test_loss'][epoch],"|",result['test_acc'][epoch])

Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  
0 | 1.836254233688846 | 0.6327300566951292 | 0.7734444353863106 | 0.7128012410731003 | 0.7734444353863106 | 0.7128012410731003
1 | 1.7005600203901587 | 0.6992847585284097 | 0.7186328579096776 | 0.7558910851365448 | 0.7186328579096776 | 0.7558910851365448
2 | 1.6697682891821288 | 0.7157898003945209 | 0.6905362731600642 | 0.7706154545697702 | 0.6905362731600642 | 0.7706154545697702
3 | 1.6547650903877742 | 0.7242354434221235 | 0.6706133087765719 | 0.7794420520501779 | 0.6706133087765719 | 0.7794420520501779
4 | 1.6456250119909128 | 0.7285875877670948 | 0.6637276112323849 | 0.7874929788429134 | 0.6637276112323849 | 0.7874929788429134
5 | 1.6391450789716169 | 0.7317672051094588 | 0.6560118670355886 | 0.7905154197983256 | 0.6560118670355886 | 0.7905154197983256
6 | 1.6345609375983772 | 0.7336156594429807 | 0.6526974515754916 | 0.7972824778666381 | 0.6526974515754916 | 0.7972824778666381
7 | 1.6311512951344693 | 0